In [ ]:
!pip install -q gradio requests python-dotenv

In [ ]:
import os, base64, requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

In [ ]:
OPENROUTER_API_KEY=your_api_key_here
API_KEY = os.getenv("OPENROUTER_API_KEY")
URL = "https://openrouter.ai/api/v1/chat/completions"

In [ ]:
MODELS = {
    "Gemini": "google/gemini-2.0-flash-001",
    "GPT": "openai/gpt-4o-mini"
}

In [ ]:
SYSTEM = """
You are a cautious AI health-information assistant.
Give general health information and safe self-care guidance.
Do not claim to be a doctor.
Do not provide definitive diagnoses or prescribe medication.
Mention warning signs and when professional medical care is needed.
For emergencies, recommend urgent medical attention.
"""

In [ ]:
def call_ai(prompt, model):
    data = {
        "model": MODELS[model],
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt}
        ]
    }

    r = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json=data
    )

    return r.json()["choices"][0]["message"]["content"]

In [ ]:
def consult(question, model):
    prompt = f"""
User health question:
{question}

Provide:
- General explanation
- Safe advice
- Warning signs
- When to seek professional care
"""
    return call_ai(prompt, model)

In [ ]:
def home_care(problem, model):
    prompt = f"""
The user asks about:
{problem}

Suggest only low-risk general home-care measures.
Explain when home care is not appropriate.
"""
    return call_ai(prompt, model)

In [ ]:
def image_analysis(image, model):
    if not image:
        return "Please upload an image."

    with open(image, "rb") as f:
        img = base64.b64encode(f.read()).decode()

    data = {
        "model": MODELS[model],
        "messages": [{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """
Describe only visible features in this image.
Do not diagnose.
Give cautious general information and recommend
professional evaluation when appropriate.
"""
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img}"
                    }
                }
            ]
        }]
    }

    r = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json=data
    )

    return r.json()["choices"][0]["message"]["content"]

In [ ]:
def get_feedback(response, model):
    prompt = f"""
Review this AI health response:

{response}

Check for:
1. Safety
2. Unsupported medical claims
3. Missing warning signs
4. Clarity

Give an improved version if necessary.
"""
    return call_ai(prompt, model)

In [ ]:
with gr.Blocks(title="AI Doctor Assistant") as app:

    gr.Markdown("# 🩺 AI Doctor Assistant")
    gr.Markdown(
        "General health information only. "
        "This does not replace professional medical care."
    )

    model = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Gemini",
        label="Choose AI Model"
    )

    with gr.Tab("🩺 Consultation"):

        question = gr.Textbox(
            label="Describe your health question",
            lines=5
        )

        consult_btn = gr.Button("Consult AI")
        consult_output = gr.Markdown()

        consult_btn.click(
            consult,
            [question, model],
            consult_output
        )

    with gr.Tab("🏠 Home Care"):

        problem = gr.Textbox(
            label="What do you need help with?",
            lines=4
        )

        care_btn = gr.Button("Get Home-Care Advice")
        care_output = gr.Markdown()

        care_btn.click(
            home_care,
            [problem, model],
            care_output
        )

    with gr.Tab("🖼️ Image Analysis"):

        image = gr.Image(
            type="filepath",
            label="Upload Image"
        )

        image_btn = gr.Button("Analyze Image")
        image_output = gr.Markdown()

        image_btn.click(
            image_analysis,
            [image, model],
            image_output
        )

    with gr.Tab("🔎 Feedback"):

        response = gr.Textbox(
            label="Paste AI Response",
            lines=8
        )

        feedback_btn = gr.Button("Check Response")
        feedback_output = gr.Markdown()

        feedback_btn.click(
            get_feedback,
            [response, model],
            feedback_output
        )

In [ ]:
app.launch()